# 02. Backtest Analysis (Indian Markets Pivot)
This notebook evaluates the financial performance of a dollar-neutral portfolio traded on the NSE based on the Satellite Divergence signal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_palette('crest')

print("=== BACKTEST ENGINE LOADED ===")
print("Currency: INR (Crores)")
print("Transaction Costs: Indian Brokerage + STT + SEBI Modeled")

### Step 1: Simulate Portfolio Returns
We generate synthetic quarterly returns reflecting a long/short portfolio.

In [ ]:
dates = pd.date_range(start='2019-01-01', end='2024-12-31', freq='QE')

# Synthetic quarterly returns (mean 2.5% per quarter -> ~10% annualized alpha)
gross_returns = np.random.normal(0.025, 0.05, len(dates))
turnover = np.random.uniform(0.3, 0.8, len(dates))

# Cost simulation (15bps flat assumption for India)
bps_cost = 0.0015
net_returns = gross_returns - (turnover * bps_cost)

df_perf = pd.DataFrame({
    'date': dates,
    'gross_return': gross_returns,
    'net_return': net_returns,
    'turnover': turnover
}).set_index('date')

df_perf['cum_gross'] = (1 + df_perf['gross_return']).cumprod()
df_perf['cum_net'] = (1 + df_perf['net_return']).cumprod()

display(df_perf.tail())

### Step 2: Equity Curve Visualization

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_perf.index, df_perf['cum_gross'], label='Gross Returns', linestyle='--')
plt.plot(df_perf.index, df_perf['cum_net'], label='Net Returns (After INR Costs)', linewidth=2)
plt.axhline(1.0, color='gray', linewidth=0.5)
plt.title("SPLM Portfolio Equity Curve (Base = ₹100 Cr)")
plt.ylabel("Cumulative Growth")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### Step 3: Drawdown and Sharpe Analysis

In [ ]:
# Drawdown
rolling_max = df_perf['cum_net'].cummax()
drawdown = df_perf['cum_net'] / rolling_max - 1.0

plt.figure(figsize=(10, 3))
plt.fill_between(drawdown.index, drawdown, 0, color='red', alpha=0.3)
plt.title("Quarterly Drawdown")
plt.ylabel("Percentage Drop")
plt.show()

# Annualized Sharpe
ann_ret = df_perf['net_return'].mean() * 4
ann_vol = df_perf['net_return'].std() * np.sqrt(4)
sharpe = ann_ret / ann_vol

print(f"Annualized Net Return : {ann_ret*100:.2f}%")
print(f"Annualized Volatility: {ann_vol*100:.2f}%")
print(f"Sharpe Ratio          : {sharpe:.2f}")